In [1]:
#@title Colab Setup Environment

try:
    import google.colab
    !mkdir -p repository && cd repository && \
     git clone https://github.com/safety-research/circuit-tracer && \
     curl -LsSf https://astral.sh/uv/install.sh | sh && \
     uv pip install -e circuit-tracer/

    import sys
    from huggingface_hub import notebook_login
    sys.path.append('repository/circuit-tracer')
    sys.path.append('repository/circuit-tracer/demos')
    notebook_login(new_session=False)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

In [2]:
from circuit_tracer import ReplacementModel, attribute
import torch

In [3]:
model_name = 'google/gemma-2-2b'
transcoder_name = "gemma"
model = ReplacementModel.from_pretrained(model_name, transcoder_name, device='cpu', dtype=torch.bfloat16)

Fetching 26 files:   0%|          | 0/26 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loaded pretrained model google/gemma-2-2b into HookedTransformer


In [4]:
prompt = "The capital of Texas is"
max_n_logits = 10
desired_logit_prob = 0.95
max_feature_nodes = 8192
batch_size=256
offload='disk' if IN_COLAB else 'cpu'
verbose = True

In [5]:
graph = attribute(
    prompt=prompt,
    model=model,
    max_n_logits=max_n_logits,
    desired_logit_prob=desired_logit_prob,
    batch_size=batch_size,
    max_feature_nodes=max_feature_nodes,
    offload=offload,
    verbose=verbose
)

Phase 0: Precomputing activations and vectors
Precomputation completed in 0.76s
Found 4337 active features
Phase 1: Running forward pass
Forward pass completed in 9.10s
Phase 2: Building input vectors
Selected 10 logits with cumulative probability 0.6406
Will include 4337 of 4337 feature nodes
Input vectors built in 0.34s
Phase 3: Computing logit attributions
C:\Users\brfie\Documents\School\COTCircuits\faithfulness\circuit_tracer\attribution\context.py:218: UserWarning: Full backward hook is firing when gradients are computed with respect to module outputs since no inputs require gradients. See https://docs.pytorch.org/docs/main/generated/torch.nn.Module.html#torch.nn.Module.register_full_backward_hook for more details.
  self._resid_activations[last_layer].backward(
Logit attributions completed in 2.59s
Phase 4: Computing feature attributions
Feature influence computation: 100%|██████████| 4337/4337 [00:16<00:00, 267.61it/s]
Feature attributions completed in 16.21s
Attribution complet

# Influence Pruning

In [6]:
from circuit_tracer.graph import prune_graph

def graph_prunings(
    graph,
    node_thresholds,
    edge_threshold = 0.98
):
    results = []
    
    n_features = len(graph.selected_features)
    n_tokens = graph.n_pos
    n_error_nodes = graph.cfg.n_layers * n_tokens

    for threshold in node_thresholds:
        print(f"Pruning with node_threshold = {threshold:.2f}")

        prune_result = prune_graph(graph, node_threshold=threshold, edge_threshold=edge_threshold)
        node_mask = prune_result.node_mask

        feature_mask = node_mask[:n_features]
        selected_feature_indices = torch.where(feature_mask)[0]
        active_feature_indices = graph.selected_features[selected_feature_indices]
        feature_nodes = graph.active_features[active_feature_indices].tolist()
        
        error_mask = node_mask[n_features : n_features + n_error_nodes]
        error_indices = torch.where(error_mask)[0]
        error_nodes = []
        for flat_idx in error_indices:
            layer = flat_idx.item() // n_tokens
            pos = flat_idx.item() % n_tokens
            error_nodes.append((layer, pos))
            
        result_entry = {
            'feature_nodes': feature_nodes,
            'error_nodes': error_nodes,
        }
        results.append(result_entry)

    return results


In [7]:
import numpy as np
pruned_graphs = graph_prunings(
    graph=graph,
    node_thresholds=np.arange(0.05, 1.0, 0.05),
)

Pruning with node_threshold = 0.05
Pruning with node_threshold = 0.10
Pruning with node_threshold = 0.15
Pruning with node_threshold = 0.20
Pruning with node_threshold = 0.25
Pruning with node_threshold = 0.30
Pruning with node_threshold = 0.35
Pruning with node_threshold = 0.40
Pruning with node_threshold = 0.45
Pruning with node_threshold = 0.50
Pruning with node_threshold = 0.55
Pruning with node_threshold = 0.60
Pruning with node_threshold = 0.65
Pruning with node_threshold = 0.70
Pruning with node_threshold = 0.75
Pruning with node_threshold = 0.80
Pruning with node_threshold = 0.85
Pruning with node_threshold = 0.90
Pruning with node_threshold = 0.95


In [8]:
pruned_graph = pruned_graphs[8]

In [9]:
with torch.inference_mode():
    new_logits, _ = model.graph_ablation(
        inputs=prompt,
        selected_features=pruned_graph['feature_nodes'],
        selected_errors=pruned_graph['error_nodes'],
        direct_effects=True
    )


torch.Size([1, 6, 2304])


In [10]:
answer = " Austin"
answer_idx = model.tokenizer(answer).input_ids[-1]

In [11]:
def metric_fn(logits):
    logits = logits.squeeze()[-1]
    answer_logit = logits[answer_idx]
    mean_top_10 = torch.topk(logits, 10).values.mean()
    return answer_logit - mean_top_10

In [12]:
metric_fn(new_logits).item()

1.125